# Getting started

How to work with model weights, layers, and activations in this project.

**Before you run anything:** check the kernel in the top-right corner reads
**`Python (research ARM64)`**. The `python3` kernel on this machine is an
emulated x64 build with no PyTorch installed, and picking it produces confusing
`ModuleNotFoundError`s.

Run the next cell first — it verifies the environment and pins model downloads
to `D:/research/models`.

In [ ]:
# `research` must be imported before `transformers` / `huggingface_hub`:
# it sets HF_HUB_CACHE, and huggingface_hub reads that once, at import time.
import research
from research import models, paths

import os
import sys

import torch

print("python     :", sys.version.split("(")[0].strip(), sys.version.split("[")[-1].rstrip("]"))
print("torch      :", torch.__version__, "| threads:", torch.get_num_threads())
print("model cache:", os.environ["HF_HUB_CACHE"])
assert "ARM64" in sys.version, "Wrong kernel - pick 'Python (research ARM64)'"

## 1. Load a model

`models.load()` returns a `(model, tokenizer)` pair on the CPU in `float32`.

Mind the memory budget: this machine has **16 GB of unified RAM and no GPU**, so
a full float32 load costs roughly **4 bytes per parameter**. That caps you at
around **1-3 B parameters**. `gpt2` (124 M) is tiny and loads in seconds; a 9 B
model will not fit at any dtype. See section 5 for how to inspect those anyway.

In [ ]:
MODEL_ID = "gpt2"          # <- the only line to change to study another model

model, tok = models.load(MODEL_ID)
OUT = paths.results_dir(MODEL_ID)          # results/<model>/ - all output goes here
print("results ->", OUT)

n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params:,} parameters ({n_params * 4 / 1e9:.2f} GB at float32)")
print("blocks:", len(models.block_names(model)))
model.config.model_type

## 2. See the layers

`models.layer_table()` lists every submodule that owns parameters, with its
class, parameter count, and the shape of each tensor. Modules with no
parameters of their own (activations, dropout) are skipped.

**Tied weights are counted once.** GPT-2 ties `lm_head.weight` to
`transformer.wte.weight` — they are the same tensor. Counting both would report
163 M parameters for a 124 M model, so the tied module is credited 0 parameters
and flagged, with `tied_to` naming the owner. The total therefore reconciles
with `sum(p.numel() for p in model.parameters())`.

In [ ]:
import pandas as pd

table = pd.DataFrame(models.layer_table(model))

print(f"{len(table)} parameterised modules")
print(f"layer_table total : {table['params'].sum():,}")
print(f"model total       : {sum(p.numel() for p in model.parameters()):,}")
print("tied:", table.loc[table["tied"], ["name", "tied_to"]].to_dict("records"))

table.to_csv(OUT / "layer-table.csv", index=False)
table.head(10)

Parameter count by module class — where the weight actually sits:

In [ ]:
table.groupby("class")["params"].agg(["count", "sum"]).sort_values("sum", ascending=False)

## 3. Read a weight tensor

`models.weight()` takes the dotted module path from the table above and returns
a detached tensor, so you can compute on it without touching autograd.

In [ ]:
ATTN = "attn|self_attn"          # matches GPT-2 and Llama/Gemma naming

first_attn, _ = models.find(models.blocks(model)[0][1], ATTN)[0]
attn_path = f"{models.block_names(model)[0]}.{first_attn}.weight"
w = models.weight(model, attn_path)
print(attn_path)
print(f"shape {tuple(w.shape)}  mean {w.mean():+.5f}  std {w.std():.5f}")

How weight scale varies with depth — a cheap first look at a real model:

In [ ]:
import matplotlib.pyplot as plt

attn_paths = [f"{b}.{models.find(m, ATTN)[0][0]}.weight"
              for b, m in models.blocks(model)]
stds = [models.weight(model, p).std().item() for p in attn_paths]

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(len(stds)), stds, marker="o")
ax.set_xlabel("layer")
ax.set_ylabel("std of c_attn.weight")
ax.set_title(f"Attention weight scale by depth ({MODEL_ID})")
ax.grid(alpha=.3)
plt.tight_layout()
fig.savefig(OUT / "attn-weight-scale-by-depth.png", dpi=150)

## 4. Capture activations

`models.capture()` is a context manager that registers forward hooks on the
modules you name and collects their outputs.

Always use it rather than calling `register_forward_hook` yourself: it removes
the hooks on exit **even if the body raises**. A leaked hook stays attached to
the model and silently corrupts every later forward pass in the session — one
of the nastier bugs to track down in a long-lived notebook.

Wrap the forward pass in `torch.no_grad()`; nothing here trains, and the
autograd graph would waste the memory that is already the binding constraint.

In [ ]:
inputs = tok("The capital of France is", return_tensors="pt")

names = models.block_names(model)
watch = [names[0], names[len(names) // 2], names[-1]]

with models.capture(model, watch) as acts:
    with torch.no_grad():
        out = model(**inputs)

for name in watch:
    a = acts[name]
    print(f"{name:<24} {str(tuple(a.shape)):<16} mean {a.mean():+.4f}  std {a.std():.4f}")

print()
print("next token ->", repr(tok.decode(out.logits[0, -1].argmax())))

Residual-stream norm growing through the network — the classic view you would
otherwise reach for TransformerLens to get. (TransformerLens does not install on
Windows ARM64; see `docs/arm64-python-stack.md`.)

In [ ]:
blocks = models.block_names(model)

with models.capture(model, blocks) as acts:
    with torch.no_grad():
        model(**inputs)

norms = [acts[b][0, -1].norm().item() for b in blocks]

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(norms, marker="o")
ax.set_xlabel("block")
ax.set_ylabel("residual norm (last token)")
ax.set_title(f"Residual stream norm by depth ({MODEL_ID})")
ax.grid(alpha=.3)
plt.tight_layout()
fig.savefig(OUT / "residual-norm-by-depth.png", dpi=150)

## 5. Inspect models too large to load

`models.raw_tensors()` reads shapes and dtypes straight from the safetensors
header. Nothing is materialised, so this works on files far larger than RAM —
this is how you study a 9 B model on a machine that cannot hold one.

In [ ]:
snapshot = next(paths.MODELS.rglob("*.safetensors")).parent
tensors = models.raw_tensors(snapshot)

print(f"{len(tensors)} tensors in {snapshot.name}")
for key in list(tensors)[:5]:
    shape, dtype, shard = tensors[key]
    print(f"  {key:<32} {str(shape):<18} {dtype}")

Pull exactly one tensor into memory when you need the values:

In [ ]:
name = next(iter(tensors))
one = models.read_tensor(snapshot / tensors[name][2], name)
print(name, tuple(one.shape), one.dtype)

### Quantised weights from Ollama

`models.gguf_tensors()` reads the GGUF blobs Ollama already has on disk, which
lets you compare a quantised copy against the original safetensors.

Note this reads the whole file and is slow — an 815 MB blob took over five
minutes. Cache the result rather than re-running the cell.

In [ ]:
from collections import Counter

blob = models.ollama_blob("gemma3:1b")
print(f"{blob.name[:24]}...  {blob.stat().st_size / 1e6:.0f} MB")

gguf = models.gguf_tensors(blob)                 # slow - see note above
print(f"{len(gguf)} tensors")
print("quantisation mix:", dict(Counter(t["quant"] for t in gguf)))

## House rules

- **Notebooks are the working surface.** Explore here, with prose explaining
  what you are testing and why.
- **Logic that a second notebook needs graduates to `src/research/`** and gets
  a test. If you are copy-pasting a cell between notebooks, it belongs in the
  package.
- **A finding that lives only in a cell is not finished** — write it up in
  `docs/`, with the model id, revision, dtype and seed needed to reproduce it.
- **Seed anything stochastic** (`torch.manual_seed`) and record the seed.
- **Clear outputs before committing**, unless the output *is* the
  documentation (as in this notebook).

Next: copy `notebooks/_template.ipynb` to start your own.